In [1]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from rasterio.features import rasterize
from rasterio.transform import from_origin
import rasterio.enums
from scipy import ndimage
from collections import Counter

In [6]:
# ============================================================
# 1 CONFIGURACIÓN Y MAPEOS DE CLASES
# ============================================================

folder_path = "/home/angelo/MyStore/Proyecto Investigador/series_temporales/series_temporales_2019_normalized(backscatter_coherence_nir)"

file_names = [
    "Alfalfa_series_2019.npz",
    "Barley_series_2019.npz",
    "Carrot_series_2019.npz",
    "Chickpea_series_2019.npz",
    "Cotton_series_2019.npz",
    "Fallow_series_2019.npz",
    "Maize_series_2019.npz",
    "Oats_series_2019.npz",
    "Onion_series_2019.npz",
    "Pepper_series_2019.npz",
    "Potato_series_2019.npz",
    "Pumpkin_series_2019.npz",
    "Quinoa_series_2019.npz",
    "Rice_series_2019.npz",
    "SugarBeet_series_2019.npz",
    "Sunflower_series_2019.npz",
    "SweetPotato_series_2019.npz",
    "Tomato_series_2019.npz",
    "Wheat_series_2019.npz",
]
nombres_cultivos = [
    "None","Alfalfa","Barley","Carrot","Chickpea","Cotton",
    "Fallow","Maize","Oats","Onion","Pepper","Potato","Pumpkin",
    "Quinoa","Rice","SugarBeet","Sunflower","SweetPotato","Tomato","Wheat"
]

nombre_a_codigo = {
    "Alfalfa":1,"Barley":2,"Carrot":3,"Chickpea":4,"Cotton":5,
    "Fallow":6,"Maize":7,"Oats":8,"Onion":9,"Pepper":10,"Potato":11,
    "Pumpkin":12,"Quinoa":13,"Rice":14,"SugarBeet":15,"Sugar beet":15,
    "Sunflower":16,"SweetPotato":17,"Sweet potato":17,"Tomato":18,"Wheat":19,
}

# ============================================================
# 2 CARGAR GT Y PREDICCIONES
# ============================================================

crop_map_npz = np.load("/home/angelo/MyStore/TFM_PAS/crop_map_2019/crop_type_map_2019.npz")
crop_map_data = crop_map_npz["data"]

def erode_crop_map(crop_map, erosion_size=3):

    if erosion_size > 1:

        structure = np.ones(
            (erosion_size, erosion_size),
            dtype=bool
        )

        crop_map_eroded = np.zeros_like(crop_map)

        for crop_code in np.unique(crop_map):

            if crop_code == 0:
                continue

            # Máscara del cultivo actual
            mask_crop = (crop_map == crop_code)

            # Erosión
            eroded_mask = ndimage.binary_erosion(
                mask_crop,
                structure=structure
            )

            # Recuperar el código del cultivo
            crop_map_eroded[eroded_mask] = crop_code

    else:

        crop_map_eroded = crop_map.copy()

    return crop_map_eroded
    
crop_map = erode_crop_map(crop_map_data, erosion_size=3)
H, W = crop_map.shape
print("crop_map shape:", crop_map.shape)

pixel_pred_model = np.load("/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/pixel_predicted_classes.npy")
pixel_pred_cropcode =  pixel_pred_model
# ============================================================
# 3 RECONSTRUIR COORDENADAS row/col
# ============================================================

coords_all = np.concatenate([
    np.load(os.path.join(folder_path,f))["coords"]
    for f in file_names
])
rows = coords_all[:,0].astype(int)
cols = coords_all[:,1].astype(int)

orig_code_px = crop_map[rows, cols]

# ============================================================
# 4 CARGAR SHP + TRANSFORM
# ============================================================

gdf = gpd.read_file("/home/angelo/MyStore/TFM_PAS/SHP2019_erosionado_filtrado/SHP_2019_erosionado_filtrado.shp")
print("CRS SHP original:", gdf.crs)
print("Parcelas totales SHP:", len(gdf))

# ============================================================
# 5 FILTRO 1: PARCELAS ≥ 1 ha
# ============================================================

parcelas_totales = len(gdf)
print("\n🔹 Parcelas totales antes del filtro 1 ha:", parcelas_totales)

gdf = gdf[gdf["area_m2"] >= 10000].copy()

print("🔹 Parcelas después del filtro 1 ha:", len(gdf))
print("   → Eliminadas por tamaño:", parcelas_totales - len(gdf))

# Crear ID único
gdf_parcelas = gdf[["burnid","DS_CULTIVO","area_m2","geometry"]].copy()

# ============================================================
# 6 CREAR GeoDataFrame DE PÍXELES + SPATIAL JOIN
# ============================================================
with rasterio.open("/home/angelo/MyStore/TFM_PAS/cropmap_2019_erosionado/cropmap_2019_erosionado.tif") as src:
    transform = src.transform
    raster_crs = src.crs


xs, ys = rasterio.transform.xy(transform, rows, cols)

pixel_gdf = gpd.GeoDataFrame(
    {"class_pred_crop":pixel_pred_cropcode,"orig_code_px":orig_code_px},
    geometry=gpd.points_from_xy(xs, ys),
    crs=raster_crs
)

pixel_parcel_joined = gpd.sjoin(
    pixel_gdf,
    gdf_parcelas,
    how="inner",
    predicate="within",
    lsuffix="px",
    rsuffix="shp"
)
print("\nPíxeles asignados:", len(pixel_parcel_joined))
# ============================================================
# 7 FILTRO 2: PARCELAS CON ≥ 10 PÍXELES
# ============================================================

px_por_parcela = pixel_parcel_joined.groupby("burnid").size()

parcelas_validas = px_por_parcela[px_por_parcela >= 10].index
parcelas_invalidas = px_por_parcela[px_por_parcela < 10].index

print("\n🔹 Parcelas con <10 píxeles:", len(parcelas_invalidas))
print("🔹 Parcelas con ≥10 píxeles:", len(parcelas_validas))

pixel_parcel_joined = pixel_parcel_joined[
    pixel_parcel_joined["burnid"].isin(parcelas_validas)
]

# ============================================================
# 8 CLASIFICACIÓN
# ============================================================

resultados = []

for burnid, group in pixel_parcel_joined.groupby("burnid"):
    # ----------------------------
    # Datos básicos de la parcela
    # ----------------------------
    total_px = len(group)
    ds_cultivo = group["DS_CULTIVO"].iloc[0]
    
    # Código original esperado según shapefile
    orig_code_expected = nombre_a_codigo.get(ds_cultivo, None)
    
    # Códigos "nuevos" (predicción) en espacio crop_map (1–19)
    new_codes = group["class_pred_crop"].astype(int).values
    
    # Si por alguna razón no hay píxeles válidos, saltamos
    if total_px == 0:
        continue
    
    conteo_new = Counter(new_codes)
    clase_mayor, freq_mayor = conteo_new.most_common(1)[0]
    porcentaje_mayor = freq_mayor / total_px  # proporción de la clase más frecuente
    
    # Fracción de píxeles cuya nueva clase coincide con lo esperado
    if orig_code_expected is not None:
        correct_count = conteo_new.get(orig_code_expected, 0)
        frac_correct = correct_count / total_px
    else:
        frac_correct = 0.0
    
    # ---------------------------------------------
    # CLASIFICACIÓN SEGÚN REGLAS DEL ARTÍCULO
    # ---------------------------------------------
    # 1) Parcela CORRECTA:
    #    Al menos el 70% de los píxeles tienen la nueva clase igual a la clase original esperada
    if (orig_code_expected is not None) and (frac_correct >= 0.70):
        tipo = "Correcta"
        nueva_clase = orig_code_expected
    
    else:
        # 2) CANDIDATA A REETIQUETADO:
        #    Más del 75% de los píxeles pertenecen a la misma nueva clase
        #    Y esta nueva clase es DIFERENTE de la original
        if (orig_code_expected is not None) and (porcentaje_mayor >= 0.70) and (clase_mayor != orig_code_expected):
            tipo = "Reetiquetable"
            nueva_clase = clase_mayor
        
        else:
            # 3) DIVISIÓN INCORRECTA:
            #    Existen al menos dos clases nuevas distintas
            #    cada una representando al menos el 40% de los píxeles
            orden_clases = sorted(conteo_new.items(), key=lambda x: x[1], reverse=True)

            clase1, freq1 = orden_clases[0]
            pct1 = freq1 / total_px
            
            clase2 = freq2 = pct2 = None
            if len(orden_clases) > 1:
                clase2, freq2 = orden_clases[1]
                pct2 = freq2 / total_px
            
            # División incorrecta: dos clases >= 40%
            clases_grandes = [c for c, f in conteo_new.items() if f / total_px >= 0.40]
            
            if len(clases_grandes) >= 2:
                tipo = "DivisionIncorrecta"
                nueva_clase = None
            
                clase1_div = clase1
                pct1_div = round(pct1 * 100, 2)
            
                clase2_div = clase2
                pct2_div = round(pct2 * 100, 2)

            
            else:
                # 4) CASO LÍMITE:
                #    Cualquier otra anomalía (no cumple reetiquetable ni división incorrecta)
                tipo = "CasoLimite"
                nueva_clase = None
    
    # ---------------------------------------------
    # Registrar resultados de la parcela
    # ---------------------------------------------
    
    
    resultados.append({
        "burnid": burnid,
        #"burnid": group["burnid"].iloc[0],
        #"DS_CULTIVO": ds_cultivo,
        "total_px": total_px,
        "orig_code_expected": orig_code_expected,
        "orig_nombre_expected": (
            nombres_cultivos[orig_code_expected] 
            if (orig_code_expected is not None and 0 <= orig_code_expected < len(nombres_cultivos)) 
            else None
        ),
        #"mayor_clase_pred": int(clase_mayor),
        "mayor_clase_pred_nombre": (
            nombres_cultivos[clase_mayor]
            if 0 <= clase_mayor < len(nombres_cultivos)
            else "Desconocido"
        ),
        "porcentaje_mayor": round(porcentaje_mayor * 100, 2),
        "frac_correct": round(frac_correct * 100, 2),
        "tipo_parcela": tipo,
        #"nueva_clase_sugerida": int(nueva_clase) if nueva_clase is not None else None,
        "nueva_clase_nombre": (
            nombres_cultivos[nueva_clase]
            if (nueva_clase is not None and 0 <= nueva_clase < len(nombres_cultivos))
            else None
        ),
        # NUEVAS COLUMNAS PARA DIVISIÓN INCORRECTA
        "clase1_div": nombres_cultivos[clase1_div] if tipo == "DivisionIncorrecta" else None,
        "pct1_div": pct1_div if tipo == "DivisionIncorrecta" else None,
        "clase2_div": nombres_cultivos[clase2_div] if tipo == "DivisionIncorrecta" else None,
        "pct2_div": pct2_div if tipo == "DivisionIncorrecta" else None,
    })
df_parcelas = pd.DataFrame(resultados)

print("\n🔹 Parcelas finalmente clasificadas:", len(df_parcelas))

# ============================================================
# 9 GUARDADO
# ============================================================

csv_out = "/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/parcelas_clasificadas_RF.csv"
df_parcelas.to_csv(csv_out, index=False)
print("CSV guardado:", csv_out)

if not df_parcelas.empty:
    
    gdf_result = gdf_parcelas.merge(df_parcelas, on="burnid", how="left")
    
    shp_out = "/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/parcelas_clasificadas_RF.shp"
    gdf_result.to_file(shp_out)
    print(f"💾 Shapefile de parcelas clasificadas guardado en: {shp_out}")
    



crop_map shape: (3587, 3607)
CRS SHP original: EPSG:32629
Parcelas totales SHP: 3920

🔹 Parcelas totales antes del filtro 1 ha: 3920
🔹 Parcelas después del filtro 1 ha: 3363
   → Eliminadas por tamaño: 557

Píxeles asignados: 1342850

🔹 Parcelas con <10 píxeles: 0
🔹 Parcelas con ≥10 píxeles: 3363

🔹 Parcelas finalmente clasificadas: 3363
CSV guardado: /home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/parcelas_clasificadas_RF.csv


/tmp/ipykernel_115828/4044219436.py:304: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_result.to_file(shp_out)
/home/angelo/.local/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'orig_code_expected' to 'orig_code_'
  ogr_write(
/home/angelo/.local/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'orig_nombre_expected' to 'orig_nombr'
  ogr_write(
/home/angelo/.local/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mayor_clase_pred_nombre' to 'mayor_clas'
  ogr_write(
/home/angelo/.local/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'porcentaje_mayor' to 'porcentaje'
  ogr_write(
/home/angelo/.local/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'frac_correct' to 'frac_corre'
  o

💾 Shapefile de parcelas clasificadas guardado en: /home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/parcelas_clasificadas_RF.shp


In [4]:
import pandas as pd

df_parcelas = pd.read_csv("/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_ndvi_nir/parcelas_clasificadas_RF.csv")

print(df_parcelas["tipo_parcela"].value_counts())

tipo_parcela
Correcta              3269
Reetiquetable           52
CasoLimite              27
DivisionIncorrecta      15
Name: count, dtype: int64


In [7]:
import pandas as pd

df_parcelas = pd.read_csv("/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_nir/parcelas_clasificadas_RF.csv")

print(df_parcelas["tipo_parcela"].value_counts())

tipo_parcela
Correcta              3253
Reetiquetable           52
CasoLimite              42
DivisionIncorrecta      16
Name: count, dtype: int64


In [3]:
import pandas as pd

df_parcelas = pd.read_csv("/home/angelo/MyStore/Proyecto Investigador/Resultados_2019_complete/RF/backscatter_coh_ndvi/parcelas_clasificadas_RF.csv")

print(df_parcelas["tipo_parcela"].value_counts())

tipo_parcela
Correcta              3254
Reetiquetable           52
CasoLimite              37
DivisionIncorrecta      20
Name: count, dtype: int64


In [3]:
import pandas as pd

df_parcelas = pd.read_csv("/home/angelo/MyStore/TFM_PAS/Resultados 2019_complete_coh/RF/parcelas_clasificadas_RF.csv")

print(df_parcelas["tipo_parcela"].value_counts())

tipo_parcela
Correcta              3214
CasoLimite              77
Reetiquetable           51
DivisionIncorrecta      21
Name: count, dtype: int64
